# College Scorecard API — Endpoint Reference

The College Scorecard API is maintained by the U.S. Department of Education.
It requires a **free API key** from [api.data.gov](https://api.data.gov).

- **Base URL:** `https://api.data.gov/ed/collegescorecard/v1/schools`
- **Docs:** https://collegescorecard.ed.gov/data/documentation/
- **Approach:** Single endpoint (`/schools`) with different field selections via the `fields` parameter.
- **Year prefix:** Use `latest.` for the most recent data, or a specific year like `2022.` for historical data.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("SCOREBOARD_API_KEY")
BASE_URL = "https://api.data.gov/ed/collegescorecard/v1/schools"

## Total Institutions

Every response includes a `metadata.total` field indicating how many institutions
match the query. An unfiltered request reveals the total number of schools in the
database.

In [ ]:
params = {
    "api_key": API_KEY,
    "fields": "school.name",
    "per_page": 1,
}

resp = requests.get(BASE_URL, params=params)
data = resp.json()

print(f"Total institutions in College Scorecard: {data['metadata']['total']}")

## 1. Search by School Name

**Filter:** `school.name`
Returns all institutions whose name contains the search string.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "Boston College",
    "fields": ",".join([
        "school.name",
        "school.city",
        "school.state",
        "latest.student.size",
    ]),
}

resp = requests.get(BASE_URL, params=params)
data = resp.json()

print(f"Matches: {data['metadata']['total']}")
pd.DataFrame(data["results"])

## 2. Admissions Data

**Fields prefix:** `latest.admissions`
Includes overall admission rate and SAT / ACT score distributions.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "Stanford University",
    "fields": ",".join([
        "school.name",
        "latest.admissions.admission_rate.overall",
        "latest.admissions.sat_scores.average.overall",
        "latest.admissions.act_scores.midpoint.cumulative",
    ]),
}

resp = requests.get(BASE_URL, params=params)
pd.DataFrame(resp.json()["results"])

## 3. Cost & Tuition

**Fields prefix:** `latest.cost`
In-state and out-of-state tuition, average net price, and average faculty salary.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "University of Michigan-Ann Arbor",
    "fields": ",".join([
        "school.name",
        "latest.cost.tuition.in_state",
        "latest.cost.tuition.out_of_state",
        "latest.cost.avg_net_price.overall",
        "latest.school.faculty_salary",
    ]),
}

resp = requests.get(BASE_URL, params=params)
pd.DataFrame(resp.json()["results"])

## 4. Post-Graduation Earnings

**Fields prefix:** `latest.earnings`
Median earnings at 6 and 10 years after entry — sourced from Treasury / IRS data.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "Massachusetts Institute of Technology",
    "fields": ",".join([
        "school.name",
        "latest.earnings.6_yrs_after_entry.median",
        "latest.earnings.10_yrs_after_entry.median",
    ]),
}

resp = requests.get(BASE_URL, params=params)
pd.DataFrame(resp.json()["results"])

## 5. Financial Aid & Debt

**Fields prefix:** `latest.aid`
Pell Grant rate, federal loan rate, and median debt for completers and non-completers.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "Harvard University",
    "fields": ",".join([
        "school.name",
        "latest.aid.pell_grant_rate",
        "latest.aid.federal_loan_rate",
        "latest.aid.median_debt.completers.overall",
        "latest.aid.median_debt.noncompleters",
    ]),
}

resp = requests.get(BASE_URL, params=params)
pd.DataFrame(resp.json()["results"])

## 6. Student Demographics

**Fields prefix:** `latest.student`
Student body size and race / ethnicity breakdowns (shares sum to ~1.0).

In [ ]:
params = {
    "api_key": API_KEY,
    "school.name": "New York University",
    "fields": ",".join([
        "school.name",
        "latest.student.size",
        "latest.student.demographics.race_ethnicity.white",
        "latest.student.demographics.race_ethnicity.black",
        "latest.student.demographics.race_ethnicity.hispanic",
        "latest.student.demographics.race_ethnicity.asian",
    ]),
}

resp = requests.get(BASE_URL, params=params)
pd.DataFrame(resp.json()["results"])

## 7. Filter by State

**Filter:** `school.state`
Two-letter abbreviation restricts results to a single state.

In [ ]:
params = {
    "api_key": API_KEY,
    "school.state": "CA",
    "fields": ",".join([
        "school.name",
        "school.city",
        "latest.student.size",
        "latest.admissions.admission_rate.overall",
    ]),
    "sort": "latest.student.size:desc",
    "per_page": 10,
}

resp = requests.get(BASE_URL, params=params)
data = resp.json()

print(f"Total institutions in California: {data['metadata']['total']}")
pd.DataFrame(data["results"])

## 8. Sorting & Pagination

**Parameters:** `sort`, `per_page`, `page`
Sort by any returned field (append `:asc` or `:desc`). Control page size with `per_page`
and navigate with `page` (0-indexed).

In [ ]:
params = {
    "api_key": API_KEY,
    "fields": ",".join([
        "school.name",
        "school.state",
        "latest.student.size",
    ]),
    "sort": "latest.student.size:desc",
    "per_page": 10,
    "page": 0,
}

resp = requests.get(BASE_URL, params=params)
data = resp.json()

print(f"Total institutions: {data['metadata']['total']}")
print(f"Page {data['metadata']['page']} — showing {data['metadata']['per_page']} per page")
pd.DataFrame(data["results"])